<a href="https://colab.research.google.com/github/CMILINAZZO/medical-llm-self-bias-audit/blob/main/notebooks/03_deepeval_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Setup & Installations
# Install DeepEval and the official SDKs for our judge models.
!pip install -q deepeval openai anthropic google-genai pandas tqdm

In [ ]:
# Cell 2: Secure Credentials
import os
from google.colab import userdata

# DeepEval automatically looks for these specific environment variables when initializing default models.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

print("API Keys successfully loaded into environment variables for DeepEval.")

In [ ]:
# Cell 3: Repository Sync & Path Setup
import sys
import shutil
from pathlib import Path
import pandas as pd
import os

# 1. Detect runtime environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Environment Detected: Google Colab")

    # Configuration
    GITHUB_USERNAME = "CMILINAZZO"
    REPO_NAME = "medical-llm-self-bias-audit"
    REPO_ROOT = Path(f"/content/{REPO_NAME}")

    # 2. Check for fake or corrupted non-git directories
    if REPO_ROOT.exists() and not (REPO_ROOT / ".git").exists():
        print("Found an empty or plain folder block where a Git repo should be. Clearing it...")
        shutil.rmtree(REPO_ROOT)

    # 3. Clean clone or pull sequence
    if not REPO_ROOT.exists():
        print(f"Cloning fresh copy of the public repository from GitHub...")
        !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
    else:
        print(f"Active Git repository found. Pulling latest upstream updates...")
        current_dir = os.getcwd()
        os.chdir(REPO_ROOT)
        !git pull
        os.chdir(current_dir)

    # 4. Synchronize Python's working directory
    os.chdir(REPO_ROOT / "notebooks")
    print(f"\n Active Working Directory synchronized to: {os.getcwd()}")

# 5. Standardized portable paths
API_DATA_PATH = "../outputs/generation_results.csv"
LOCAL_DATA_PATH = "../outputs/generation_results_local.csv"

# Load both datasets
df_api = pd.read_csv(API_DATA_PATH)
df_local = pd.read_csv(LOCAL_DATA_PATH)

# Merge the local open-weights columns (Llama and Gemma) into the main dataframe
# Assuming the base rows (pmid, context, question) align perfectly
df = df_api.copy()
df['response_llama'] = df_local['response_llama']
df['response_gemma'] = df_local['response_gemma']

print(f"Datasets merged successfully. Matrix ready with {df.shape[0]} rows and {df.shape[1]} columns.")

In [ ]:
# Sanity Check
# Cell 4: Data Merge Sanity Check
print("--- MERGE SANITY CHECK ---")

# 1. Check for expected column presence
expected_cols = ['response_gpt4o', 'response_claude', 'response_gemini', 'response_llama', 'response_gemma']
missing_cols = [col for col in expected_cols if col not in df.columns]

if missing_cols:
    print(f"ERROR: Missing columns after merge: {missing_cols}")
else:
    print("All 5 student response columns are successfully merged.")

# 2. Check for unexpected nulls (which would indicate misaligned rows)
null_counts = df[expected_cols].isnull().sum()
if null_counts.sum() > 0:
    print("WARNING: Found null values in student responses!")
    print(null_counts[null_counts > 0])
else:
    print("No null values detected in the student columns.")

# 3. Visual check of the top row
print("\nFirst row student model keys available:")
print(df[expected_cols].head(1).to_dict(orient='records')[0].keys())

In [ ]:
from deepeval.metrics import FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from tqdm import tqdm
from deepeval.models import AnthropicModel, GeminiModel

# Cell 5: Define Judges
# We rely on out-of-the-box support to avoid complex prompt tuning.
# Temperature is strictly locked at 0.0 inside DeepEval's execution by default for these models.
claude_judge = AnthropicModel(model="claude-sonnet-4-6", temperature=0)
gemini_judge = GeminiModel(model="gemini-2.5-pro", temperature=0)

JUDGE_MODELS = [
    "gpt-4o",       # DeepEval routes OpenAI strings natively
    claude_judge,
    gemini_judge
]

# The Student models we want to evaluate from our dataframe
STUDENT_COLS = [
    'response_gpt4o',
    'response_claude',
    'response_gemini',
    'response_llama',
    'response_gemma'
]

print("DeepEval framework and judge matrix initialized.")

In [ ]:
# Cell 6: Judging Loop
# This loop processes the student answers through the judges to measure hallucination (Faithfulness) and clinical accuracy (Correctness).
# IMPORTANT NOTE: Don't run this as is unless you are operating with a highly-funded API tier!

from deepeval.metrics import FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCase
from tqdm import tqdm

results = []

print("Commencing the DeepEval Audit Matrix...")

# Loop through each Judge Model
for judge_name in JUDGE_MODELS:
    # Use string name for logging if it's an object
    display_name = judge_name if isinstance(judge_name, str) else judge_name.model
    print(f"\n Activating Judge: {display_name}")

    # Initialize metrics with the current judge
    faithfulness_metric = FaithfulnessMetric(threshold=0.5, model=judge_name, include_reason=False)
    correctness_metric = GEval(name="Correctness", model=judge_name, criteria="Determine whether the actual output is factually correct based on the expected output.",
                               evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT], threshold=0.5)

    for idx, row in tqdm(df.iterrows(), total=df.shape[0], desc=f"Evaluating with {display_name}"):
        for student_col in STUDENT_COLS:
            test_case = LLMTestCase(
                input=row['question'],
                actual_output=str(row[student_col]),
                expected_output=row['ground_truth'],
                retrieval_context=[row['context']]
            )

            try:
                faithfulness_metric.measure(test_case)
                correctness_metric.measure(test_case)

                results.append({
                    'pmid': row['pmid'],
                    'student_model': student_col.replace('response_', ''),
                    'judge_model': display_name,
                    'faithfulness_score': faithfulness_metric.score,
                    'correctness_score': correctness_metric.score
                })
            except Exception as e:
                print(f"Error evaluating {student_col} with {display_name} on row {idx}: {e}") # This is too broad if you're broke, see below.

# Convert to final matrix
df_audit = pd.DataFrame(results)

print("\n Evaluation cycle complete. Sanity check of the final matrix:")
print("-" * 50)
print(df_audit.head())

### The Messy Reality: API Limits & Salvaging a Crashed Loop

**Note:** If you are operating with a highly-funded API tier, the judging loop above (Cell 6) will likely finish smoothly.

However, in my original run, DeepEval's multi-step extractions triggered severe API rate limits and a hard billing timeout mid-execution. Instead of restarting the loop from scratch (and paying for duplicate API calls), I engineered the following two cells to extract the partial data from active memory and perform a budget-conscious "Smart Resume".

*If Cell 6 completed successfully for you, you can safely skip/comment out Cells 7 and 8.*

In [ ]:
# Cell 7: Salvage Partial Data (Fixed for Sorting)
import pandas as pd

df_partial = pd.DataFrame(results)
df_partial = df_partial.dropna(subset=['faithfulness_score', 'correctness_score'])

# 1. Force the column to be strings
df_partial['judge_model'] = df_partial['judge_model'].astype(str)

# 2. Clean up the ugly object names for our matrix
def clean_judge_name(name):
    if 'anthropic' in name.lower(): return 'claude-sonnet-4-6'
    if 'google' in name.lower(): return 'gemini-2.5-pro'
    return name

df_partial['judge_model'] = df_partial['judge_model'].apply(clean_judge_name)

# 3. Save to disk so we don't lose it
df_partial.to_csv("../outputs/salvaged_audit_matrix.csv", index=False)

print(f"Salvaged {len(df_partial)} successful evaluations!")
display(df_partial.groupby('judge_model').size())

In [ ]:
# IMPORTANT NOTE: This did not work perfectly for me, but rather than try to re-engineer it, I decided to cut my losses.
# The exception was too broad, and so it didn't bring everything to a halt when encountering errors telling me that I was out of funds.
# I should have fixed that here, but didn't realize until after I had run the script.
# The "Smart Resume" logic turned out to be "Smart" because DeepEval leaked raw memory addresses (like <anthropic.Anthropic object...>) into the judge_model column.
# Because it is only comparing the name of the judge in the CSV against the name of the judge in the loop, it just fired the API again when it encountered the leaked raw memory addresses.
# So, if you're going to run this, definitely address those issues first.

# Cell 8: The 50-Row "Smart" Resume Audit
from deepeval.metrics import FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from tqdm import tqdm
import pandas as pd
import os

print("Commencing the 50-Row Smart Resume Audit...")

# 1. Isolate the 50-Row target dataset
df_50 = df.head(50).copy()

# 2. Load the salvaged data so we don't pay for duplicates
salvage_path = "../outputs/salvaged_audit_matrix.csv"
if os.path.exists(salvage_path):
    df_salvaged = pd.read_csv(salvage_path)
    print(f"Loaded {len(df_salvaged)} previously salvaged evaluations.")
else:
    df_salvaged = pd.DataFrame()
    print("No salvaged data found. Starting from scratch.")

results = []

for judge_name in JUDGE_MODELS:
    display_name = judge_name if isinstance(judge_name, str) else judge_name.model
    print(f"\n Activating Judge: {display_name}")

    # Initialize metrics
    f_metric = FaithfulnessMetric(threshold=0.5, model=judge_name, include_reason=False)
    c_metric = GEval(
        name="Correctness",
        model=judge_name,
        criteria="Determine whether the actual output is factually correct based on the expected output.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.5
    )

    for idx, row in tqdm(df_50.iterrows(), total=df_50.shape[0], desc=f"Evaluating with {display_name}"):
        for student_col in STUDENT_COLS:
            student_clean_name = student_col.replace('response_', '')

            # 3. Check if we already did this exact pair
            if not df_salvaged.empty:
                already_done = df_salvaged[
                    (df_salvaged['pmid'] == row['pmid']) &
                    (df_salvaged['student_model'] == student_clean_name) &
                    (df_salvaged['judge_model'] == display_name)
                ]

                if not already_done.empty:
                    # Append the old data and skip the API call
                    results.append(already_done.iloc[0].to_dict())
                    continue

            # 4. If not done, call the API
            test_case = LLMTestCase(
                input=row['question'],
                actual_output=str(row[student_col]),
                expected_output=row['ground_truth'],
                retrieval_context=[row['context']]
            )

            try:
                f_metric.measure(test_case)
                c_metric.measure(test_case)

                results.append({
                    'pmid': row['pmid'],
                    'student_model': student_clean_name,
                    'judge_model': display_name,
                    'faithfulness_score': f_metric.score,
                    'correctness_score': c_metric.score
                })
            except Exception as e:
                print(f"Error on row {idx} ({student_col}): {e}")

# Convert to the final 50-row matrix
df_audit = pd.DataFrame(results)

print("\n 50-Row Evaluation cycle complete!")
print("-" * 50)
print(f"Final Matrix Size: {df_audit.shape[0]} rows (Should be exactly 750)")

### Resuming Standard Execution
*(End of the salvage operation!)*

Whether you successfully completed the full matrix in Cell 6, or used the Smart Resume in Cell 8, you should now have a fully populated `df_audit` DataFrame. The next step is calculating our deterministic statistical baselines.

In [ ]:
# Cell 9: Traditional Statistical Baseline (ROUGE-L)
!pip install -q rouge-score

from rouge_score import rouge_scorer
import numpy as np

print("Calculating non-LLM baseline metrics (ROUGE-L)...")
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# Dictionary to cache scores so we only calculate them once per student answer
rouge_cache = {}

def get_rouge_l(row):
    cache_key = (row['pmid'], row['student_model'])

    # 1. Check if we've already calculated this exact pair
    if cache_key in rouge_cache:
        return rouge_cache[cache_key]

    # 2. If not, retrieve the texts and calculate
    ground_truth = df.loc[df['pmid'] == row['pmid'], 'ground_truth'].values[0]
    student_response = df.loc[df['pmid'] == row['pmid'], f"response_{row['student_model']}"].values[0]

    score = scorer.score(str(ground_truth), str(student_response))
    f1_score = score['rougeL'].fmeasure

    # 3. Save to cache and return
    rouge_cache[cache_key] = f1_score
    return f1_score

# Apply the metric to every row in our final matrix
df_audit['rougeL_f1_baseline'] = df_audit.apply(get_rouge_l, axis=1)

print("ROUGE-L baseline calculated efficiently and added to the audit matrix.")

In [ ]:
# Cell 10: Self-Healing Export to Permanent Storage
import os

OUTPUT_PATH = "../outputs/final_audit_matrix.csv"
output_dir = os.path.dirname(OUTPUT_PATH)

os.makedirs(output_dir, exist_ok=True)
df_audit.to_csv(OUTPUT_PATH, index=False)

print(f"Final 15-pair matrix written successfully to: {OUTPUT_PATH}")

### Final Step: Save Your Matrix

Google Colab virtual machines are temporary. If you close this tab, your `final_audit_matrix.csv` will be deleted.

To permanently save your progress:
1. Open the **Files** sidebar (folder icon on the left).
2. Navigate into the `outputs/` folder.
3. Right-click `final_audit_matrix.csv` and select **Download**.
4. (Optional) If you linked this notebook to GitHub, use Colab's native menu: **File -> Save a copy in GitHub**.

## LLM-Based Metrics
Using an LLM to evaluate hallucination and semantic meaning introduces inherent risk, especially in an audit specifically looking for LLM bias.  

**Pros:**
* The faithfulness metric uses a standardized Question-Answer-Generation (QAG) protocol to mathematically score interactions (rather than zero-shot grading).
* The correctness metric operates on a Chain-of-Thought (CoT) and claim-extraction methodology (rather than zero-shot grading).
* Out-of-the-Box functionality avoids complex prompt engineering on our end.

**Cons:**
* Because an LLM is extracting the claims and verifying them, its own internal safety alignment (or self-criticism) can taint the extraction phase before the scoring even happens.
* Highly conservative models might flag complex medical jargon paraphrasing as a contradiction, when a human doctor would recognize it as semantically valid.

## Risk Mitigation Strategy
Run deterministic, non-LLM statistical metrics alongside the LLM judges.
1. ROUGE-L or BLEU Scores
2. Cosine Similarity (e.g. BERTScore)

## Performance & Architecture Decision Note: Sequential Execution vs. Parallel Processing

Evaluating 1,500 distinct student-judge pairs sequentially introduces an I/O bottleneck (resulting in a 60-90 minute execution runtime). It is possible to make the process faster by using multithreading. However, I intentionally avoided multithreading to avoid crossing API rate-limiting thresholds.

DeepEval decomposes evaluation metrics into multi-step operations under the hood, multiplying the number of required underlying API requests. Because this project is running on Tier-1 commercial balances and free developer quotas, bursting concurrent requests would instantly trigger `429 Too Many Requests` HTTP errors.